<a href="https://colab.research.google.com/github/Zainab-Binte-Khalid/urdu-ocr-codesaviours-si26-zainab/blob/main/SI26_Week_04_Zainab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from torch.utils.data import Dataset
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd
import os
import re

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, image_root, processor):
        self.data = pd.read_csv(csv_path)
        self.image_root = image_root
        self.processor = processor
        self.file_lookup = {}
        for category in os.listdir(image_root):
            cat_path = os.path.join(image_root, category)
            if os.path.isdir(cat_path):
                for fname in os.listdir(cat_path):
                    match = re.search(r"(\d+)(?=\.\w+$)", fname)
                    if match:
                        num = int(match.group(1))
                        self.file_lookup[(category, num)] = fname

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        rel_path = row["image"]
        if rel_path.startswith("images/"):
            rel_path = rel_path[len("images/"):]
        parts = rel_path.split("/")
        category = parts[0]
        csv_filename = parts[-1]
        image_path = os.path.join(self.image_root, category, csv_filename)
        if not os.path.exists(image_path):
            match = re.search(r"(\d+)(?=\.\w+$)", csv_filename)
            if match:
                num = int(match.group(1))
                actual_fname = self.file_lookup.get((category, num))
                if actual_fname:
                    image_path = os.path.join(self.image_root, category, actual_fname)
        image = Image.open(image_path).convert("RGB")
        encoding = self.processor(image, return_tensors="pt")
        pixel_values = encoding.pixel_values.squeeze()
        labels = self.processor.tokenizer(row["text"], padding="max_length", max_length=128).input_ids
        labels = torch.tensor(labels)
        return {"pixel_values": pixel_values, "labels": labels}

print('Dataset class defined directly in notebook')

Dataset class defined directly in notebook


In [4]:
!pip uninstall transformers -y -q
!pip install transformers==4.41.2 torch pillow pandas sentencepiece -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 126.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install transformers==4.41.2 torch pillow pandas sentencepiece -q

In [3]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cpu':
    print('WARNING: No GPU detected.')
    print('Go to Runtime > Change runtime type > GPU')

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')
model = model.to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print('Model loaded successfully!')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Model parameters: 333,921,792


In [4]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import os
import re

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, image_root, processor):
        self.data = pd.read_csv(csv_path)
        self.image_root = image_root
        self.processor = processor
        self.exact_lookup = {}
        self.fuzzy_lookup = {}
        for category in os.listdir(image_root):
            cat_path = os.path.join(image_root, category)
            if os.path.isdir(cat_path):
                for fname in os.listdir(cat_path):
                    full_path = os.path.join(cat_path, fname)
                    self.exact_lookup[fname] = full_path
                    match = re.match(r'([a-zA-Z_]+?)_?(\d+)(\.\w+)$', fname)
                    if match:
                        prefix, num, ext = match.group(1), int(match.group(2)), match.group(3)
                        self.fuzzy_lookup[(prefix.rstrip('_'), num)] = full_path
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def _resolve_path(self, csv_image_path):
        basename = os.path.basename(csv_image_path)
        if basename in self.exact_lookup:
            return self.exact_lookup[basename]
        match = re.match(r'([a-zA-Z_]+?)_?(\d+)(\.\w+)$', basename)
        if match:
            prefix, num, ext = match.group(1), int(match.group(2)), match.group(3)
            key = (prefix.rstrip('_'), num)
            if key in self.fuzzy_lookup:
                return self.fuzzy_lookup[key]
            for known_prefix, known_num in list(self.fuzzy_lookup.keys()):
                if known_num == num and (known_prefix in prefix or prefix in known_prefix):
                    return self.fuzzy_lookup[(known_prefix, known_num)]
        return None

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_path = self._resolve_path(row['image'])
        image = Image.open(image_path).convert('RGB')
        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()
        labels = self.processor.tokenizer(
            row['text'], padding='max_length', max_length=128, truncation=True
        ).input_ids
        labels = torch.tensor(labels)
        return {'pixel_values': pixel_values, 'labels': labels, 'source': row['source']}

project_root = '/content/drive/MyDrive/Urdu_OCR'

dataset = UrduOCRDataset(
    csv_path=f'{project_root}/labels_final_2633.csv',
    image_root=f'{project_root}/data/processed',
    processor=processor
)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

print(f'Training samples: {train_size}, Testing samples: {test_size}')

Dataset loaded: 2633 samples
Training samples: 2106, Testing samples: 527


In [5]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

optimizer = AdamW(model.parameters(), lr=3e-5)
num_epochs = 15
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)

print(f'Training batches per epoch: {len(train_loader)}')
print('Ready to train!')

Training batches per epoch: 527
Ready to train!


In [6]:
import os
checkpoint_dir = '/content/drive/MyDrive/Urdu_OCR/checkpoints_v2'
os.makedirs(checkpoint_dir, exist_ok=True)

for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0
    print(f'\nEpoch {epoch}/{num_epochs} | LR: {scheduler.get_last_lr()[0]:.2e}')
    print('-' * 30)
    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        if batch_idx % 100 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')

    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch} complete | Average Loss: {avg_loss:.4f}')

    epoch_save_path = f'{checkpoint_dir}/epoch_{epoch}'
    model.save_pretrained(epoch_save_path)
    processor.save_pretrained(epoch_save_path)
    print(f'Checkpoint saved: {epoch_save_path}')

print('\nTraining complete!')


Epoch 1/15 | LR: 3.00e-05
------------------------------
  Batch 0/527 | Loss: 18.2348
  Batch 100/527 | Loss: 0.4552
  Batch 200/527 | Loss: 1.1262
  Batch 300/527 | Loss: 0.8337
  Batch 400/527 | Loss: 0.5523
  Batch 500/527 | Loss: 1.0266
Epoch 1 complete | Average Loss: 1.0592
Checkpoint saved: /content/drive/MyDrive/Urdu_OCR/checkpoints_v2/epoch_1

Epoch 2/15 | LR: 2.97e-05
------------------------------
  Batch 0/527 | Loss: 0.6338
  Batch 100/527 | Loss: 0.5863
  Batch 200/527 | Loss: 1.1025
  Batch 300/527 | Loss: 0.8616
  Batch 400/527 | Loss: 0.8348
  Batch 500/527 | Loss: 0.5653
Epoch 2 complete | Average Loss: 0.7247
Checkpoint saved: /content/drive/MyDrive/Urdu_OCR/checkpoints_v2/epoch_2

Epoch 3/15 | LR: 2.87e-05
------------------------------
  Batch 0/527 | Loss: 0.7774
  Batch 100/527 | Loss: 0.4947
  Batch 200/527 | Loss: 0.2554
  Batch 300/527 | Loss: 0.4344
  Batch 400/527 | Loss: 0.5966
  Batch 500/527 | Loss: 0.3869
Epoch 3 complete | Average Loss: 0.5597
Checkpo

In [7]:
from collections import defaultdict

model.eval()
correct = 0
total = 0
total_char_similarity = 0
category_stats = defaultdict(lambda: {'correct': 0, 'total': 0, 'char_sim': 0})
wrong_examples = []

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels']
        sources = batch['source']

        generated_ids = model.generate(pixel_values, max_new_tokens=128)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
        actual_text = processor.batch_decode(labels, skip_special_tokens=True)

        for pred, actual, src in zip(generated_text, actual_text, sources):
            total += 1
            pred_clean = pred.strip()
            actual_clean = actual.strip()
            is_correct = pred_clean == actual_clean
            if is_correct:
                correct += 1
            else:
                if len(wrong_examples) < 5:
                    wrong_examples.append((pred_clean, actual_clean, src))

            char_sim = 0
            if len(actual_clean) > 0:
                common = sum(1 for a, b in zip(pred_clean, actual_clean) if a == b)
                char_sim = common / max(len(actual_clean), 1)
                total_char_similarity += char_sim

            category_stats[src]['total'] += 1
            category_stats[src]['char_sim'] += char_sim
            if is_correct:
                category_stats[src]['correct'] += 1

accuracy = (correct / total) * 100 if total > 0 else 0
avg_char_sim = (total_char_similarity / total) * 100 if total > 0 else 0

print(f'Overall Exact-match Accuracy: {accuracy:.1f}% ({correct}/{total})')
print(f'Overall Character overlap: {avg_char_sim:.1f}%')
print('\n=== Per-category breakdown ===')
for cat, stats in category_stats.items():
    cat_acc = (stats['correct'] / stats['total']) * 100 if stats['total'] > 0 else 0
    cat_char = (stats['char_sim'] / stats['total']) * 100 if stats['total'] > 0 else 0
    print(f'{cat}: {cat_acc:.1f}% exact match, {cat_char:.1f}% char overlap ({stats["total"]} samples)')

print('\n=== 5 wrong examples ===')
for pred, actual, src in wrong_examples:
    print(f'[{src}] Predicted: {pred}')
    print(f'[{src}] Actual: {actual}')
    print()

Overall Exact-match Accuracy: 23.0% (121/527)
Overall Character overlap: 60.0%

=== Per-category breakdown ===
natural_scene_dataset: 43.7% exact match, 72.0% char overlap (206 samples)
clean_printed_dataset: 10.5% exact match, 53.0% char overlap (294 samples)
newspaper: 0.0% exact match, 8.6% char overlap (3 samples)
handwritten: 0.0% exact match, 59.9% char overlap (15 samples)
synthetic: 0.0% exact match, 38.1% char overlap (7 samples)
signboard: 0.0% exact match, 4.5% char overlap (2 samples)

=== 5 wrong examples ===
[natural_scene_dataset] Predicted: ابب
[natural_scene_dataset] Actual: اب

[natural_scene_dataset] Predicted: سلے
[natural_scene_dataset] Actual: سے

[natural_scene_dataset] Predicted: جارہ
[natural_scene_dataset] Actual: چارہ

[clean_printed_dataset] Predicted: اپنی گزشتہ کارو روز کی قیمت فروخت,77.80 روپے سے
[clean_printed_dataset] Actual: اپنی گزشتہ کاروباری روز کی قیمت فروخت 88.17 روپے سے

[clean_printed_dataset] Predicted: کے قرضوں میں اضافے کا شمل جاری ہے اور سمب

In [8]:
from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/SI26-urdu-ocr-model'
model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print(f'Model saved to Google Drive: {save_path}')
print('You can load this model again next week without retraining')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model saved to Google Drive: /content/drive/MyDrive/SI26-urdu-ocr-model
You can load this model again next week without retraining
